# Baseline Ablation Test
Compare baseline model performance with vs without attack/defense work-rate features.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

pd.set_option('display.max_columns', None)

In [2]:
# Load reordered dataset
df = pd.read_csv('../Final Dataset/cleaned_player_data_performance_model_reordered.csv')

target = 'overall_rating'
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

correlations = df[numeric_cols].corr()[target].drop(target).sort_values(ascending=False)
important_features = correlations[correlations.abs() > 0.1].index.tolist()

print('Selected important features:')
for i, f in enumerate(important_features, 1):
    print(f'{i:2d}. {f}')

Selected important features:
 1. potential
 2. reactions
 3. ball_control
 4. dribbling
 5. stamina
 6. strength
 7. acceleration
 8. balance
 9. defensive_work_rate_encoded
10. attacking_work_rate_encoded


In [1]:
def run_linear_regression(features, label):
    X = df[features]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    return {
        'Variant': label,
        'Num_Features': len(features),
        'Train_R2': r2_score(y_train, y_train_pred),
        'Test_R2': r2_score(y_test, y_test_pred),
        'Train_RMSE': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'Test_RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'Train_MAE': mean_absolute_error(y_train, y_train_pred),
        'Test_MAE': mean_absolute_error(y_test, y_test_pred),
        'Overfit_Diff': abs(r2_score(y_train, y_train_pred) - r2_score(y_test, y_test_pred))
    }

In [4]:
# Variant A: with attack+defense work-rate features
features_with = important_features.copy()

# Variant B: without attack+defense work-rate features
drop_cols = {'attacking_work_rate_encoded', 'defensive_work_rate_encoded'}
features_without = [f for f in important_features if f not in drop_cols]

res_with = run_linear_regression(features_with, 'With Attack+Defense')
res_without = run_linear_regression(features_without, 'Without Attack+Defense')

results = pd.DataFrame([res_with, res_without])
results = results.sort_values(['Test_R2', 'Test_RMSE'], ascending=[False, True]).reset_index(drop=True)

print('Ablation Results:')
print(results[['Variant', 'Num_Features', 'Test_R2', 'Test_RMSE', 'Test_MAE', 'Overfit_Diff']].to_string(index=False))

winner = results.iloc[0]
print('\nRecommended version to keep:')
print(f"{winner['Variant']} | Test R2={winner['Test_R2']:.6f} | Test RMSE={winner['Test_RMSE']:.6f}")

Ablation Results:
               Variant  Num_Features  Test_R2  Test_RMSE  Test_MAE  Overfit_Diff
   With Attack+Defense            10 0.838759   2.395559  1.854920      0.004285
Without Attack+Defense             8 0.836605   2.411510  1.862244      0.004031

Recommended version to keep:
With Attack+Defense | Test R2=0.838759 | Test RMSE=2.395559
